In [4]:
import torch, numpy as np
x = torch.arange(4.0, requires_grad=True)
x

tensor([0., 1., 2., 3.], requires_grad=True)

In [5]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [ ]:
y.backward() # y = 2 * (x0^2 + x1^2 + x2^2 + x3^2)，对每个 xi 求导：dy/dxi = 4xi
x.grad

tensor([ 0.,  4.,  8., 12.])

In [7]:
x.grad == 4 * x

tensor([True, True, True, True])

In [ ]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值，清理成tensor([0., 0., 0., 0.])
x.grad.zero_() 
y = x.sum() #y = x0 + x1 + x2 + x3
y.backward()
x.grad

tensor([1., 1., 1., 1.])

In [14]:
# 对非标量调用backward需要传入一个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递一个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward(retain_graph=True)
x.grad

tensor([0., 2., 4., 6.])

In [15]:
x.grad.zero_()
#y=x*x
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])

In [ ]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c
a = torch.randn(size=(), requires_grad=True)
print('a =', a)
d = f(a)
print('d =', d)
d.backward() #backward只能用于标量，若d不是标量，则需要传入一个gradient参数
a.grad == d / a

a = tensor(1.0069, requires_grad=True)
d = tensor(1031.0371, grad_fn=<MulBackward0>)


tensor(True)

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2
y.backward(gradient=torch.tensor([1.0, 10.0, 100.0]))
print(x.grad)
# 等价于对这个东西求导：
# 1 * y0 + 10 * y1 + 100 * y2
# 也就是：
# 1 * x0^2 + 10 * x1^2 + 100 * x2^2
# 所以梯度是：
# [1 * 2x0, 10 * 2x1, 100 * 2x2]
# 当 x = [1, 2, 3]：
# [2, 40, 600]

tensor([  2.,  40., 600.])


## Exercise: plot sin(x) and its derivative with autograd

Goal: define `f(x) = sin(x)`, then plot both `f(x)` and `df(x)/dx`. The derivative curve is computed by PyTorch autograd, not by manually writing `cos(x)`.

The key line is:

```python
df_dx = torch.autograd.grad(
    outputs=f_x,
    inputs=x,
    grad_outputs=torch.ones_like(f_x)
)[0]
```

- `outputs=f_x`: the result we differentiate. Here `f_x = sin(x)`.
- `inputs=x`: the variable we differentiate with respect to.
- `grad_outputs=torch.ones_like(f_x)`: 这个是因为 f_x 不是一个单独的数，而是一整个向量：
x = torch.linspace(..., 200)
f_x = torch.sin(x)
所以 f_x 有 200 个值：
[f(x1), f(x2), f(x3), ..., f(x200)]
PyTorch 不能直接对“一个向量”求一个简单梯度，它需要你告诉它：这些输出怎么合成一个总量。

- torch.ones_like(f_x)
相当于说：
每个 f(xi) 的权重都是 1，
也就是求：
f(x1) + f(x2) + ... + f(x200)
对 x 的梯度。
因为每个 f(xi) 只依赖自己的 xi，所以结果就是每个点上的导数：
[df(x1)/dx1, df(x2)/dx2, ..., df(x200)/dx200]
- `[0]`: `torch.autograd.grad(...)` returns a tuple. Since there is only one input, the first item is the derivative with respect to `x`.


In [ ]:
import math
import torch
import matplotlib.pyplot as plt

# Create x values and ask PyTorch to track gradients for x.
x = torch.linspace(-2 * math.pi, 2 * math.pi, 200, requires_grad=True)

# Define f(x) = sin(x). Do not manually use cos(x).
f_x = torch.sin(x)

# outputs=f_x: differentiate this output.
# inputs=x: compute the derivative with respect to x.
# grad_outputs all ones: same as differentiating f_x.sum().
# The return value is a tuple, so [0] extracts the derivative for x.
df_dx = torch.autograd.grad(
    outputs=f_x,
    inputs=x,
    grad_outputs=torch.ones_like(f_x)
)[0]

# detach() removes gradient tracking before passing tensors to matplotlib.
x_np = x.detach().numpy()
f_np = f_x.detach().numpy()
df_np = df_dx.detach().numpy()

plt.figure(figsize=(7, 4))
plt.plot(x_np, f_np, label='f(x) = sin(x)')
plt.plot(x_np, df_np, label='df(x)/dx by autograd')
plt.xlabel('x')
plt.ylabel('value')
plt.grid(True)
plt.legend()
plt.show()
